In [1]:
library(Seurat)
library(Signac)
library(GenomeInfoDb)
library(EnsDb.Hsapiens.v86)
library(ggplot2)
library(patchwork)
library(hdf5r)
library(future)
library(RColorBrewer)
library(dplyr)
library(Matrix)
library(BSgenome.Hsapiens.UCSC.hg38)
library(glue)
library(harmony)
library(matrixStats)
library(scales)
library(biomaRt)
library(curl)
library(goseq)
library(httr)
library(Scillus)
library(TFBSTools)
library(JASPAR2020)
library(ggridges)
library(ggrepel)
library(ggsignif)
library(qusage)
library(tidyverse)
library(effsize)
library(DESeq2)
httr::set_config(config(ssl_verifypeer = 0L))
set.seed(1234)
setwd("/home/jupyter/scATAC_analysis/edit/snatac-rcc-manuscript")
source("scripts/functions.r")


Attaching SeuratObject

Loading required package: BiocGenerics


Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, intersect, is.unsorted, lapply, Map, mapply,
    match, mget, order, paste, pmax, pmax.int, pmin, pmin.int,
    Position, rank, rbind, Reduce, rownames, sapply, setdiff, sort,
    table, tapply, union, unique, unsplit, which.max, which.min


Loading required package: S4Vectors

Loading required package: stats4


Attaching package: ‘S4Vectors’


The following object is masked from ‘package:utils’:

    findMatches


The following objects are masked from ‘package:base’:

    expand.grid, I, unname


Loading required package: IRanges

Loading required package: ensembldb

Loading required packag

# Table S4

## Sheet A-B: Linear mixed effects model results for genomic regions whose accessibility is associated with IFN1 and IFNG cytosig scores in multiome ATAC-RNA tumor cells

In [ ]:
ifn1_me_sig <- readRDS("processed_data/IFN1_peak_signature.rds")
ifn1_me_sig[, 5] <- NULL
write.table(ifn1_me_sig, "tables/s4a_IFN1_peak_set.txt", sep = "\t", quote = F, col.names = T, row.names = F)


In [ ]:
ifng_me_sig <- readRDS("processed_data/IFNG_peak_signature.rds")
ifng_me_sig[, 5] <- NULL
write.table(ifng_me_sig, "tables/s4b_IFNG_peak_set.txt", sep = "\t", quote = F, col.names = T, row.names = F)


## Sheet C: Pathways enriched in IFN epigenetic signatures via GREAT

In [2]:
results <- list()

ifn1_great_results <- readRDS("processed_data/ifn1_GREAT_pathway_results.rds")
ifn1_great_results$group <- "IFN1"
results[["IFN1"]] <- ifn1_great_results

ifng_great_results <- readRDS("processed_data/ifng_GREAT_pathway_results.rds")
ifng_great_results$group <- "IFNG"
results[["IFNG"]] <- ifng_great_results

write.table(dplyr::bind_rows(results), "tables/s4c_IFN1_IFNG_GREAT_GOBP_MSigDB_output.txt", sep = "\t", quote = F, row.names = F, col.names = T)


## Sheet D-E: TF binding site motif enrichment in IFN1 and IFNG epigenetic signatures

In [ ]:
ifn1_enriched.motifs <- readRDS("processed_data/IFN1_tfmotifs.rds")


write.table(ifn1_enriched.motifs %>% filter(p.adjust < 0.05),
    file = "tables/s4d_IFN1_tfmotifs.txt", sep = "\t", quote = F, row.names = F, col.names = T
)


In [ ]:
ifng_enriched.motifs <- readRDS("processed_data/IFNG_tfmotifs.rds")

write.table(ifng_enriched.motifs %>% filter(p.adjust < 0.05),
    file = "tables/s4e_IFNG_tfmotifs.txt", sep = "\t", quote = F, row.names = F, col.names = T
)


## Sheet F: Linear mixed effects model results for ERVs whose accessibility is associated with IFN1 and IFNG epigenetic signature scores in ccRCC balanced tumor cells

In [ ]:
ifn1_results <- readRDS("processed_data/ifn1_ervs_mixedeffects.rds")
ifn1_sig_results <- ifn1_results %>% filter((p.adjust < 0.05) & (Estimate > 0))

ifng_results <- readRDS("processed_data/ifng_ervs_mixedeffects.rds")
ifng_sig_results <- ifng_results %>% filter((p.adjust < 0.05) & (Estimate > 0))


In [ ]:
ifn1_sig_results$group <- "IFN1"
ifng_sig_results$group <- "IFNG"

ifn_sig_results <- dplyr::bind_rows(list(ifn1_sig_results, ifng_sig_results))
write.table(ifn_sig_results, "tables/s4f_erv_ifn_associations.txt", sep = "\t", quote = F, row.names = F, col.names = T)


## Sheet G: Associations between IFN-associated ERVs and BAP1 mutation status in scATAC-seq tumor cells

In [ ]:
erv_results <- readRDS("processed_data/ifnerv_bap1_wilcox_cliff.rds")

supp_table <- erv_results %>% dplyr::select(!c("lo", "hi", "sig"))
write.table(supp_table, "tables/s4g_ifnerv_bap1_wilcox_cliff.txt", sep = "\t", row.names = F, quote = F)


# Export to Excel

In [1]:
import pandas as pd
import os
os.chdir('/home/jupyter/scATAC_analysis/edit/snatac-rcc-manuscript')

In [2]:
ifn1_peaks = pd.read_csv('tables/s4a_IFN1_peak_set.txt', sep = '\t')
ifn1_peaks.head()

ifng_peaks = pd.read_csv('tables/s4b_IFNG_peak_set.txt', sep = '\t')
ifng_peaks.head()

ifn_pathways = pd.read_csv('tables/s4c_IFN1_IFNG_GREAT_GOBP_MSigDB_output.txt', sep = '\t')
ifn_pathways.head()

ifn1_tf_motifs = pd.read_csv('tables/s4d_IFN1_tfmotifs.txt', sep = '\t')
ifn1_tf_motifs.head()

ifng_tf_motifs = pd.read_csv('tables/s4e_IFNG_tfmotifs.txt', sep = '\t')
ifng_tf_motifs.head()

ifn_assoc_ervs = pd.read_csv('tables/s4f_erv_ifn_associations.txt', sep = '\t')
ifn_assoc_ervs['feature'] = ifn_assoc_ervs['feature'].str.replace('.', '_', regex=True)
ifn_assoc_ervs.head()

bap1_ifn_ervs = pd.read_csv('tables/s4g_ifnerv_bap1_wilcox_cliff.txt', sep = '\t')
bap1_ifn_ervs = bap1_ifn_ervs.rename(columns = {'delta': 'Cliffs Delta'})

In [3]:
with pd.ExcelWriter('tables/table_S4_draft.xlsx') as writer:  
    ifn1_peaks.to_excel(writer, sheet_name='A', index = False)
    ifng_peaks.to_excel(writer, sheet_name='B', index = False)
    ifn_pathways.to_excel(writer, sheet_name='C', index = False)
    ifn1_tf_motifs.to_excel(writer, sheet_name='D', index = False)
    ifng_tf_motifs.to_excel(writer, sheet_name='E', index = False)
    ifn_assoc_ervs.to_excel(writer, sheet_name='F', index = False)
    bap1_ifn_ervs.to_excel(writer, sheet_name='G', index = False)

Add README with titles in excel app. 